# Chapter 2: CLIP — Connecting Images and Text

*Build a Multimodal Model from Scratch*

---

## Chapter Goal

Build **CLIP** (Contrastive Language–Image Pre-training) from scratch.

CLIP is the crucial bridge between image and language modalities.  After
training, CLIP maps images and their descriptions to the *same point* in a
shared embedding space.  This enables:

* **Zero-shot image classification** — classify images using free-form text labels.
* **Image search** — retrieve images by text query.
* **VLM backbone** — the image encoder in LLaVA, InstructBLIP, and GPT-4V.

By the end of this chapter you will understand:
1. Why the standard ViT image-classification pre-training is insufficient for VLMs.
2. The **InfoNCE contrastive loss** — the mathematical heart of CLIP.
3. The role of the **learnable temperature parameter τ**.
4. How to perform **zero-shot classification** with a trained CLIP model.

In [ ]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

os.makedirs('figures', exist_ok=True)
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

---
## 2.1  Why Not Just Use ImageNet Pre-training?

The ViT we built in Chapter 1 was trained to classify images into fixed
categories (e.g., 1,000 ImageNet classes).  That works well for its
training distribution, but has two critical limitations for VLMs:

1. **Closed vocabulary** — the model only knows ImageNet categories.
   If an image contains something not in those 1,000 classes, the features
   are meaningless.

2. **No language alignment** — the internal representation of "a photo of a
   cat" has no geometric relationship to the word-embedding of "cat" inside
   a language model.  You cannot simply concatenate them.

CLIP solves both problems by training jointly on (image, text) pairs.
Instead of predicting a fixed class label, CLIP learns to predict:
*which text caption matches which image in a batch?*

The training signal is the full diversity of natural language — not a
fixed 1,000-class vocabulary.  After training, image embeddings and text
embeddings live in the same space.

---
## 2.2  The CLIP Architecture: Dual Encoder

CLIP has two encoders — one for images, one for text — that project into
a shared embedding space.  The figure below shows the architecture.

In [ ]:
def draw_clip_architecture():
    from matplotlib.patches import FancyBboxPatch
    BLUE   = '#4A90D9'
    PURPLE = '#9B59B6'
    GREEN  = '#5CB85C'
    DARK   = '#2C3E50'
    WHITE  = '#FFFFFF'

    def _box(ax, x, y, w, h, color, text, fs=10):
        p = FancyBboxPatch((x-w/2, y-h/2), w, h,
                           boxstyle='round,pad=0.05,rounding_size=0.3',
                           facecolor=color, edgecolor=DARK, linewidth=1.2, zorder=3)
        ax.add_patch(p)
        ax.text(x, y, text, ha='center', va='center', fontsize=fs,
                color=WHITE, fontweight='bold', zorder=4)

    def _arrow(ax, x1, y1, x2, y2):
        ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle='->', color=DARK, lw=1.5), zorder=2)

    fig, ax = plt.subplots(figsize=(13, 6))
    ax.set_xlim(0, 13); ax.set_ylim(0, 6)
    ax.axis('off'); fig.patch.set_facecolor('#F8F9FA')

    imgs_left = ['Image of a cat', 'Image of a car', 'Forest scene']
    bg_left   = ['#E8F4FD', '#FEF9E7', '#EAFAF1']
    for i, (cap, col) in enumerate(zip(imgs_left, bg_left)):
        y = 5.0 - i*1.5
        ax.add_patch(FancyBboxPatch((0.2, y-0.35), 2.0, 0.7,
                     boxstyle='round,pad=0.05', facecolor=col, edgecolor=DARK))
        ax.text(1.2, y, cap, ha='center', va='center', fontsize=9)
        _arrow(ax, 2.3, y, 2.9, y)

    _box(ax, 3.5, 3.25, 1.2, 4.2, BLUE, 'Vision\nEncoder\n(ViT)', 10)

    for i in range(3):
        y = 5.0 - i*1.5
        _arrow(ax, 4.1, y, 4.7, y)
        _box(ax, 5.1, y, 0.8, 0.5, BLUE, f'I{i+1}', 9)

    texts_right = ['a photo of a cat', 'a photo of a car', 'photo of a forest']
    for i, txt in enumerate(texts_right):
        y = 5.0 - i*1.5
        ax.add_patch(FancyBboxPatch((7.5, y-0.35), 2.5, 0.7,
                     boxstyle='round,pad=0.05', facecolor='#F4ECF7', edgecolor=PURPLE))
        ax.text(8.75, y, f'"{txt}"', ha='center', va='center', fontsize=8.5, style='italic')
        _arrow(ax, 7.5, y, 7.1, y)

    _box(ax, 6.5, 3.25, 1.2, 4.2, PURPLE, 'Text\nEncoder', 10)

    sim_ax = ax.inset_axes([0.72, 0.08, 0.26, 0.84])
    sim = np.array([[0.9,0.1,0.1],[0.1,0.85,0.15],[0.05,0.1,0.92]])
    sim_ax.imshow(sim, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')
    sim_ax.set_xticks([0,1,2]); sim_ax.set_yticks([0,1,2])
    sim_ax.set_xticklabels(['T1','T2','T3'], fontsize=8)
    sim_ax.set_yticklabels(['I1','I2','I3'], fontsize=8)
    sim_ax.set_title('Similarity\nMatrix', fontsize=9, fontweight='bold')
    for i in range(3):
        for j in range(3):
            sim_ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                        fontsize=8, fontweight='bold',
                        color='white' if sim[i,j] > 0.5 else DARK)

    ax.text(11.0, 0.35, 'Goal: diagonal HIGH\noff-diagonal LOW',
            ha='center', va='center', fontsize=9,
            bbox=dict(facecolor='white', edgecolor=GREEN, boxstyle='round', lw=1.5))
    ax.set_title('CLIP: Contrastive Language-Image Pre-training',
                 fontsize=13, fontweight='bold', color=DARK, pad=10)
    plt.tight_layout()
    plt.savefig('figures/ch02_clip_arch.png', dpi=120, bbox_inches='tight')
    plt.show()

draw_clip_architecture()

---
## 2.3  InfoNCE Contrastive Loss

### 2.3.1  The Intuition

Consider a batch of `B` (image, text) pairs.  The model computes embeddings
for all images and all texts, then scores every possible (image, text)
combination.

We want **high similarity** for matched pairs (diagonal) and **low similarity**
for mismatched pairs (off-diagonal).

### 2.3.2  Mathematical Form

For a batch of size `B`, let `s_{i,j}` be the cosine similarity between
image `i` and text `j`, divided by a temperature `τ`:

```
s_{i,j} = (image_i · text_j) / τ
```

The loss for image `i` is a cross-entropy over all texts, where label `i`
is correct:

```
L_img(i) = -log  exp(s_{i,i}) / Σ_j exp(s_{i,j})
```

We compute this symmetrically — images predicting texts AND texts predicting
images — and average:

```
L_InfoNCE = [ L_img + L_txt ] / 2
```

This is exactly `F.cross_entropy` applied twice!

### 2.3.3  Why "Contrastive"?

The loss creates two forces in embedding space:
* **Attraction**: matched pairs (same index) are pulled together.
* **Repulsion**: mismatched pairs (different indices) are pushed apart.

After training, images and text that describe the same thing cluster at
the same point in the embedding space.

In [ ]:
# ── Step-by-step InfoNCE, before wrapping it into a function ────────────────

torch.manual_seed(0)
B = 4             # batch size
D = 8             # embedding dimension (tiny for illustration)
tau = 0.07        # temperature

# Simulated embeddings after L2 normalization
img_emb = F.normalize(torch.randn(B, D), dim=-1)
txt_emb = F.normalize(torch.randn(B, D), dim=-1)

# Step 1: similarity matrix
sim = img_emb @ txt_emb.T     # (B, B)
print(f'Similarity matrix shape: {sim.shape}')

# Step 2: scale by temperature (lower τ → sharper distribution)
sim_scaled = sim / tau

# Step 3: correct labels are on the diagonal
labels = torch.arange(B)      # [0, 1, 2, 3]

# Step 4: cross-entropy from both directions
loss_img = F.cross_entropy(sim_scaled,   labels)
loss_txt = F.cross_entropy(sim_scaled.T, labels)
loss     = (loss_img + loss_txt) / 2

print(f'Loss image->text : {loss_img.item():.4f}')
print(f'Loss text->image : {loss_txt.item():.4f}')
print(f'InfoNCE loss     : {loss.item():.4f}')
print(f'Random baseline  : {math.log(B):.4f}  (= ln({B}))')

In [ ]:
def infonce_loss(img_emb, txt_emb, temperature=0.07):
    """
    Symmetric InfoNCE / NT-Xent loss.

    img_emb, txt_emb: (B, D) L2-normalised embeddings.
    Returns scalar loss.  Random baseline = ln(B).
    """
    B      = img_emb.shape[0]
    sim    = img_emb @ txt_emb.T / temperature   # (B, B)
    labels = torch.arange(B, device=img_emb.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2

print('infonce_loss defined.')

### 2.3.4  The Temperature Parameter τ

Temperature controls how *sharp* the similarity distribution is:

| τ | Effect |
|---|--------|
| τ → 0 | Distribution collapses to argmax (extreme confidence) |
| τ = 0.07 | CLIP's default; sharp but not extreme |
| τ → ∞ | Uniform distribution; random performance |

In CLIP, τ is **learnable**: it starts at 0.07 and the model adjusts it.
PyTorch stores `log(1/τ)` = `logit_scale` for numerical stability:

```python
self.logit_scale = nn.Parameter(torch.tensor(math.log(1/0.07)))
temperature = 1 / self.logit_scale.exp()
```

In [ ]:
def draw_temperature_effect():
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    torch.manual_seed(1)
    B = 8; D = 16
    img_emb = F.normalize(torch.randn(B, D), dim=-1)
    txt_emb = F.normalize(torch.randn(B, D), dim=-1)
    # make pairs 0 and 1 very similar
    txt_emb[0] = img_emb[0] + 0.05 * torch.randn(D)
    txt_emb[1] = img_emb[1] + 0.05 * torch.randn(D)
    txt_emb = F.normalize(txt_emb, dim=-1)

    sim_raw = (img_emb @ txt_emb.T).detach().numpy()

    for ax, tau, title in [
        (axes[0], 0.05, 'tau = 0.05 (very sharp)'),
        (axes[1], 0.10, 'tau = 0.10 (CLIP default)'),
        (axes[2], 0.50, 'tau = 0.50 (soft)'),
    ]:
        softmax_sim = np.exp(sim_raw / tau)
        softmax_sim /= softmax_sim.sum(axis=1, keepdims=True)
        im = ax.imshow(softmax_sim, cmap='Blues', vmin=0, vmax=1, aspect='equal')
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.set_xlabel('Text idx'); ax.set_ylabel('Image idx')
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.suptitle('Effect of Temperature tau on Similarity Distribution',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ch02_temperature.png', dpi=110, bbox_inches='tight')
    plt.show()

draw_temperature_effect()

---
## 2.4  InfoNCE Intuition: Embedding Space Before and After Training

After training, matching (image, text) pairs should cluster at the same
location in the shared embedding space.

In [ ]:
def draw_infonce_intuition():
    DARK   = '#2C3E50'
    BLUE   = '#4A90D9'
    GREEN  = '#5CB85C'
    ORANGE = '#F0AD4E'
    RED    = '#D9534F'

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    fig.patch.set_facecolor('#F8F9FA')
    np.random.seed(7)
    n = 4
    colors = [BLUE, GREEN, ORANGE, RED]
    labels_pts = ['cat', 'car', 'dog', 'bird']

    img_before = np.random.randn(n, 2) * 0.8
    txt_before = np.random.randn(n, 2) * 0.8
    centers    = np.array([[1.5, 1.5], [-1.5, 1.5], [1.5, -1.5], [-1.5, -1.5]])
    img_after  = centers + np.random.randn(n, 2) * 0.15
    txt_after  = centers + np.random.randn(n, 2) * 0.15

    for ax, ie, te, title in [
        (axes[0], img_before, txt_before, 'Before Training (random embeddings)'),
        (axes[1], img_after,  txt_after,  'After Training (matched pairs aligned)'),
    ]:
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
        ax.axhline(0, color='#BDC3C7', lw=0.8); ax.axvline(0, color='#BDC3C7', lw=0.8)
        ax.set_facecolor('#FAFAFA')
        ax.set_title(title, fontsize=11, fontweight='bold')
        for i in range(n):
            ax.plot([ie[i,0], te[i,0]], [ie[i,1], te[i,1]],
                    color=colors[i], lw=1.5, alpha=0.5, linestyle='--')
            ax.scatter(*ie[i], s=120, color=colors[i], marker='s',
                       zorder=5, edgecolors=DARK, lw=1.2)
            ax.scatter(*te[i], s=120, color=colors[i], marker='o',
                       zorder=5, edgecolors=DARK, lw=1.2)
            ax.text(ie[i,0]+0.12, ie[i,1]+0.12, f'[img]{labels_pts[i]}', fontsize=8)
            ax.text(te[i,0]+0.12, te[i,1]+0.12, f'[txt]{labels_pts[i]}', fontsize=8)

    axes[0].scatter([], [], marker='s', color=DARK, label='Image embed')
    axes[0].scatter([], [], marker='o', color=DARK, label='Text embed')
    axes[0].legend(fontsize=9)
    axes[0].set_xlabel('Embed dim 1'); axes[0].set_ylabel('Embed dim 2')
    axes[1].set_xlabel('Embed dim 1')

    plt.suptitle('InfoNCE Loss: Pull Matches Together, Push Others Apart',
                 fontsize=12, fontweight='bold', color=DARK)
    plt.tight_layout()
    plt.savefig('figures/ch02_infonce.png', dpi=120, bbox_inches='tight')
    plt.show()

draw_infonce_intuition()

---
## 2.5  Building the CLIP Model

We need two components:
1. `CLIPImageEncoder` — a ViT whose CLS output is projected and L2-normalized.
2. `CLIPTextEncoder` — a bidirectional Transformer that takes the embedding
   at the `[EOS]` token position.

Both encoders project to the same `proj_dim` dimension before L2 normalization.

In [ ]:
# ── Re-use basic building blocks from Chapter 1 ─────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class PositionalEmbedding(nn.Module):
    def __init__(self, n_patches, embed_dim):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def forward(self, x): return x + self.pos_embed

class BidirectionalAttention(nn.Module):
    """Multi-head self-attention without a causal mask (used by both ViT and CLIPTextEncoder)."""
    def __init__(self, embed_dim, n_heads, dropout=0.0):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = embed_dim // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv      = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.drop     = nn.Dropout(dropout)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attn = F.softmax((q @ k.transpose(-2,-1)) * self.scale, dim=-1)
        return self.out_proj((self.drop(attn) @ v).transpose(1,2).reshape(B,N,C))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = BidirectionalAttention(embed_dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        h          = int(embed_dim * mlp_ratio)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, h), nn.GELU(),
            nn.Linear(h, embed_dim), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.ffn(self.norm2(x))

print('Building blocks defined.')

In [ ]:
class CLIPImageEncoder(nn.Module):
    """
    ViT image encoder with a final linear projection to proj_dim.
    Output: L2-normalized vector of shape (B, proj_dim).
    """
    def __init__(self, img_size=32, patch_size=8, in_channels=3,
                 embed_dim=128, depth=4, n_heads=4, proj_dim=64):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = PositionalEmbedding(n_patches, embed_dim)
        self.blocks    = nn.ModuleList([TransformerBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm      = nn.LayerNorm(embed_dim)
        self.proj      = nn.Linear(embed_dim, proj_dim, bias=False)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        B = x.shape[0]
        t = self.patch_embed(x)
        t = torch.cat([self.cls_token.expand(B,-1,-1), t], dim=1)
        t = self.pos_embed(t)
        for blk in self.blocks: t = blk(t)
        t = self.norm(t)
        return F.normalize(self.proj(t[:, 0, :]), dim=-1)  # CLS -> proj -> L2 norm


class CLIPTextEncoder(nn.Module):
    """
    Bidirectional Transformer text encoder.
    Uses the embedding at the EOS token position (given by eos_pos).
    Output: L2-normalized vector of shape (B, proj_dim).

    Note: bidirectional (not causal) because CLIP wants the full context
    of the caption, not a prediction task.  GPT-style causal masking
    would prevent earlier tokens from seeing the full sentence meaning.
    """
    def __init__(self, vocab_size=128, context_len=32, embed_dim=128,
                 depth=4, n_heads=4, proj_dim=64):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(context_len, embed_dim)
        self.blocks    = nn.ModuleList([TransformerBlock(embed_dim, n_heads) for _ in range(depth)])
        self.norm      = nn.LayerNorm(embed_dim)
        self.proj      = nn.Linear(embed_dim, proj_dim, bias=False)

    def forward(self, token_ids, eos_positions):
        B, T = token_ids.shape
        device = token_ids.device
        x = self.tok_embed(token_ids) + \
            self.pos_embed(torch.arange(T, device=device)).unsqueeze(0)
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        # Extract the EOS token embedding for each sequence
        eos_out = x[torch.arange(B, device=device), eos_positions]
        return F.normalize(self.proj(eos_out), dim=-1)


class CLIP(nn.Module):
    def __init__(self, img_size=32, patch_size=8, vocab_size=128,
                 context_len=32, embed_dim=128, depth=4, n_heads=4, proj_dim=64):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(img_size, patch_size, 3,
                                              embed_dim, depth, n_heads, proj_dim)
        self.text_encoder  = CLIPTextEncoder(vocab_size, context_len,
                                             embed_dim, depth, n_heads, proj_dim)
        # Learnable temperature: store log(1/tau) for numerical stability
        # Initial value: log(1/0.07) ≈ 2.66
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / 0.07)))

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, token_ids, eos_positions):
        return self.text_encoder(token_ids, eos_positions)

    def forward(self, images, token_ids, eos_positions):
        img_emb = self.encode_image(images)
        txt_emb = self.encode_text(token_ids, eos_positions)
        temperature = 1.0 / self.logit_scale.exp().clamp(min=1e-8)
        loss = infonce_loss(img_emb, txt_emb, temperature=temperature.item())
        sim  = img_emb @ txt_emb.T
        return {'loss': loss, 'sim': sim, 'img_emb': img_emb, 'txt_emb': txt_emb}

print('CLIPImageEncoder, CLIPTextEncoder, CLIP defined.')

---
## 2.6  Training Demo: Color–Text Alignment

We create a minimal dataset: 4 solid-color images paired with their color names.

| Class | Image | Text |
|-------|-------|------|
| 0 | Solid red | `"red"` |
| 1 | Solid blue | `"blue"` |
| 2 | Solid green | `"green"` |
| 3 | Solid yellow | `"yellow"` |

We use a character-level tokenizer that maps ASCII characters to integer IDs.
This is intentionally simple — the focus is on showing that InfoNCE loss
drives image and text embeddings to align.

In [ ]:
def make_color_clip_dataset(n_per_class=100, img_size=32, context_len=16, seed=0):
    torch.manual_seed(seed)
    color_rgb   = [[1.0,0.1,0.1], [0.1,0.1,1.0], [0.1,0.8,0.1], [0.9,0.8,0.0]]
    color_words = ['red', 'blue', 'green', 'yellow']

    all_imgs, all_tokens, all_eos, all_labels = [], [], [], []
    for label, (rgb, word) in enumerate(zip(color_rgb, color_words)):
        char_ids = [ord(c) % 128 for c in word]
        eos_pos  = len(char_ids)
        padded   = char_ids + [1] + [0] * (context_len - len(char_ids) - 1)
        for _ in range(n_per_class):
            img = torch.tensor(rgb).reshape(3,1,1).expand(3, img_size, img_size).clone()
            img += torch.randn_like(img) * 0.06
            all_imgs.append(img.clamp(0, 1))
            all_tokens.append(torch.tensor(padded, dtype=torch.long))
            all_eos.append(eos_pos)
            all_labels.append(label)

    return (torch.stack(all_imgs), torch.stack(all_tokens),
            torch.tensor(all_eos), torch.tensor(all_labels))

imgs, tokens, eos_positions, labels = make_color_clip_dataset()
dataset = TensorDataset(imgs, tokens, eos_positions, labels)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
print(f'Images: {imgs.shape}  Tokens: {tokens.shape}  Classes: {labels.unique().tolist()}')

In [ ]:
clip = CLIP(img_size=32, patch_size=8, vocab_size=128, context_len=16,
           embed_dim=64, depth=3, n_heads=4, proj_dim=32).to(DEVICE)
optimizer = torch.optim.AdamW(clip.parameters(), lr=3e-4, weight_decay=0.01)

losses, taus = [], []
clip.train()
EPOCHS = 25
print(f'Training CLIP for {EPOCHS} epochs ...')
for epoch in range(EPOCHS):
    epoch_loss = 0
    for xb_img, xb_tok, xb_eos, _ in loader:
        xb_img = xb_img.to(DEVICE)
        xb_tok = xb_tok.to(DEVICE)
        xb_eos = xb_eos.to(DEVICE)
        out = clip(xb_img, xb_tok, xb_eos)
        optimizer.zero_grad()
        out['loss'].backward()
        optimizer.step()
        epoch_loss += out['loss'].item()
    avg = epoch_loss / len(loader)
    tau = (1.0 / clip.logit_scale.exp()).item()
    losses.append(avg); taus.append(tau)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1:3d}/{EPOCHS}  loss={avg:.4f}  tau={tau:.4f}')

print(f'\nFinal loss: {losses[-1]:.4f}  (random baseline: ln(32) = {math.log(32):.4f})')

In [ ]:
def plot_clip_results(losses, taus):
    clip.eval()
    color_names = ['red', 'blue', 'green', 'yellow']
    with torch.no_grad():
        rep_imgs   = torch.stack([imgs[labels==i][0] for i in range(4)]).to(DEVICE)
        rep_toks   = torch.stack([tokens[labels==i][0] for i in range(4)]).to(DEVICE)
        rep_eos    = torch.tensor([int(eos_positions[labels==i][0]) for i in range(4)]).to(DEVICE)
        img_emb    = clip.encode_image(rep_imgs)
        txt_emb    = clip.encode_text(rep_toks, rep_eos)
        sim_matrix = (img_emb @ txt_emb.T).cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    axes[0].plot(range(1, EPOCHS+1), losses, 'b-o', markersize=4, lw=2)
    axes[0].axhline(math.log(32), color='red', linestyle='--', alpha=0.7,
                    label=f'Random baseline ln(32)={math.log(32):.2f}')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('InfoNCE Loss')
    axes[0].set_title('CLIP Training Loss', fontweight='bold')
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

    axes[1].plot(range(1, EPOCHS+1), taus, 'g-o', markersize=4, lw=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Temperature tau')
    axes[1].set_title('Learned Temperature tau', fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)

    im = axes[2].imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='equal')
    axes[2].set_xticks(range(4)); axes[2].set_yticks(range(4))
    axes[2].set_xticklabels(color_names); axes[2].set_yticklabels(color_names)
    axes[2].set_xlabel('Text'); axes[2].set_ylabel('Image')
    axes[2].set_title('Image-Text Similarity Matrix', fontweight='bold')
    for i in range(4):
        for j in range(4):
            axes[2].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                         fontsize=9, fontweight='bold',
                         color='white' if abs(sim_matrix[i,j]) > 0.5 else '#333')
    plt.colorbar(im, ax=axes[2], fraction=0.046)

    plt.suptitle('CLIP Training Results', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ch02_training.png', dpi=120, bbox_inches='tight')
    plt.show()

plot_clip_results(losses, taus)

---
## 2.7  Zero-Shot Classification: CLIP's Most Remarkable Ability

After training, we can classify images into *new categories* using free-form
text — without any retraining.

**How it works:**

1. Encode the test image → `img_emb`.
2. For each candidate label, encode `"a photo of a {label}"` → `txt_emb_k`.
3. The predicted class is the label with the **highest cosine similarity**
   to `img_emb`.

This is called zero-shot because the model was never trained with explicit
class labels — only (image, text) pairs with InfoNCE loss.

In [ ]:
def zero_shot_classify(clip_model, image, class_words, context_len=16):
    """
    Classify `image` into one of `class_words` using zero-shot CLIP similarity.

    image      : (1, C, H, W) tensor
    class_words: list of strings, e.g. ['red', 'blue', 'green', 'yellow']
    returns    : predicted class index and similarity scores
    """
    clip_model.eval()
    with torch.no_grad():
        img_emb = clip_model.encode_image(image)  # (1, proj_dim)

        sims = []
        for word in class_words:
            char_ids = [ord(c) % 128 for c in word]
            eos_pos  = len(char_ids)
            padded   = char_ids + [1] + [0] * (context_len - len(char_ids) - 1)
            tok_ids  = torch.tensor([padded], dtype=torch.long, device=image.device)
            eos_t    = torch.tensor([eos_pos], device=image.device)
            txt_emb  = clip_model.encode_text(tok_ids, eos_t)
            sims.append((img_emb * txt_emb).sum().item())

    return sims.index(max(sims)), sims


# Test zero-shot on all 4 classes
color_rgb   = [[1.0,0.1,0.1], [0.1,0.1,1.0], [0.1,0.8,0.1], [0.9,0.8,0.0]]
color_names = ['red', 'blue', 'green', 'yellow']

print('Zero-shot classification results:')
print(f'{"Image":<8}  {"Predicted":<10}  {"Correct?":<8}  Similarities')
print('-' * 60)
for true_label, (rgb, name) in enumerate(zip(color_rgb, color_names)):
    img = torch.tensor(rgb).reshape(3,1,1).expand(3,32,32).clone().unsqueeze(0).to(DEVICE)
    pred, sims = zero_shot_classify(clip, img, color_names)
    sim_str = '  '.join(f'{n}:{s:.2f}' for n, s in zip(color_names, sims))
    correct = 'YES' if pred == true_label else 'NO'
    print(f'{name:<8}  {color_names[pred]:<10}  {correct:<8}  {sim_str}')

---
## 2.8  Chapter Summary

| Concept | Key Idea |
|---------|----------|
| **Why CLIP** | ImageNet pre-training gives a closed vocabulary; CLIP learns from free-form text |
| **Dual encoder** | CLIPImageEncoder + CLIPTextEncoder both project to the same `proj_dim` space |
| **InfoNCE loss** | Cross-entropy matching in both directions: images→texts and texts→images |
| **Temperature τ** | Scales similarity before softmax; lower τ = sharper = more confident |
| **Learnable τ** | Stored as `logit_scale = log(1/τ)`; model adjusts sharpness during training |
| **L2 normalization** | Both encoders normalize their output; similarity = cosine similarity |
| **Zero-shot** | Query: which text label has the highest cosine similarity to the image? |

**Next — Chapter 3: VLM Architecture**

CLIP gives us a shared embedding space, but CLIP cannot *generate text*.
Chapter 3 shows how to connect the CLIP image encoder to a GPT decoder
to build a model that can answer questions about images.